# RQ1 — Notebook 3: Method Redundancy Analysis

**Research Question**: Are any of the 6 clustering methods redundant, and which can be removed?

This notebook provides the data to decide whether to keep all 6 methods in the benchmark.
Three analyses are run:

1. **6×6 Pearson correlation matrix** of LSE values across all datasets — highly correlated
   methods produce similar rankings and add little diversity.
2. **Wins distribution** — how often each method is the true argmax-best.
3. **Marginal contribution to oracle** — how much mean oracle LSE drops if a method is removed.
   A method contributing < 0.01 mean LSE is a candidate for removal.

**Decision point**: Results are printed and saved as figures. The decision to drop methods
should be made here before proceeding to meta-feature extraction.

**Outputs**: `outputs/figures/method_correlation.png`, `method_wins.png`, `method_marginal.png`

In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

ROOT     = os.path.abspath(os.path.join(os.getcwd(), '..'))
META_DIR = os.path.join(ROOT, 'data', 'meta_table')
FIGS_DIR = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGS_DIR, exist_ok=True)

LSE_COLS     = ['LSE_kmeans', 'LSE_dbscan', 'LSE_agg', 'LSE_gmm', 'LSE_autoenc', 'LSE_dictlearn']
METHOD_NAMES = ['kmeans', 'dbscan', 'agg', 'gmm', 'autoenc', 'dictlearn']

df = pd.read_csv(os.path.join(META_DIR, 'meta_training.csv'))
print(f'Loaded meta_training.csv: {df.shape}')
print(f'Datasets: {len(df)}')

Loaded meta_training.csv: (86, 9)
Datasets: 86


## Analysis 1: Method Correlation Matrix

Pearson correlation of LSE values across all datasets. Methods that are highly correlated
(r > 0.85) produce nearly identical rankings and one could be removed without information loss.

In [2]:
corr_matrix = df[LSE_COLS].corr(method='pearson')
corr_matrix.index   = METHOD_NAMES
corr_matrix.columns = METHOD_NAMES

print('=== 6x6 Pearson Correlation of LSE values ===')
print(corr_matrix.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, ax=ax,
    mask=np.zeros_like(corr_matrix, dtype=bool),  # show full matrix
)
ax.set_title('LSE Correlation Between Methods\n(r > 0.85 = potentially redundant)', fontsize=11)
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'method_correlation.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'Saved → {path}')

# Highlight high-correlation pairs
print('\n=== High-correlation pairs (|r| > 0.70) ===')
for i in range(len(METHOD_NAMES)):
    for j in range(i+1, len(METHOD_NAMES)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.70:
            print(f'  {METHOD_NAMES[i]:10s} vs {METHOD_NAMES[j]:10s}  r={r:.3f}')

=== 6x6 Pearson Correlation of LSE values ===
           kmeans  dbscan    agg    gmm  autoenc  dictlearn
kmeans      1.000   0.676  0.959  0.680    0.843      0.684
dbscan      0.676   1.000  0.650  0.657    0.719      0.729
agg         0.959   0.650  1.000  0.681    0.846      0.670
gmm         0.680   0.657  0.681  1.000    0.781      0.700
autoenc     0.843   0.719  0.846  0.781    1.000      0.718
dictlearn   0.684   0.729  0.670  0.700    0.718      1.000
Saved → c:\MLResearch\outputs\figures\method_correlation.png

=== High-correlation pairs (|r| > 0.70) ===
  kmeans     vs agg         r=0.959
  kmeans     vs autoenc     r=0.843
  dbscan     vs autoenc     r=0.719
  dbscan     vs dictlearn   r=0.729
  agg        vs autoenc     r=0.846
  gmm        vs autoenc     r=0.781
  gmm        vs dictlearn   r=0.700
  autoenc    vs dictlearn   r=0.718


## Analysis 2: Wins Distribution

For each method, how many datasets have it as the argmax-best? A method that never wins
or wins very rarely contributes little to the overall benchmark.

In [3]:
wins = df['best_method'].value_counts().reindex(METHOD_NAMES, fill_value=0)
wins_pct = (wins / len(df) * 100).round(1)

print('=== Wins distribution ===')
for m, w, p in zip(wins.index, wins.values, wins_pct.values):
    bar = '█' * int(p / 2)
    print(f'  {m:10s}  {w:3d}/{len(df)}  ({p:4.1f}%)  {bar}')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4C72B0' if w > 5 else '#999999' for w in wins.values]
ax.bar(wins.index, wins.values, color=colors)
ax.set_ylabel('Number of datasets where method is best')
ax.set_title('Method Wins Distribution\n(grey = fewer than 5 wins — candidate for removal)')
for i, (m, w) in enumerate(zip(wins.index, wins.values)):
    ax.text(i, w + 0.3, str(w), ha='center', fontsize=10)
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'method_wins.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'\nSaved → {path}')

=== Wins distribution ===
  kmeans       26/86  (30.2%)  ███████████████
  dbscan        6/86  ( 7.0%)  ███
  agg           7/86  ( 8.1%)  ████
  gmm          29/86  (33.7%)  ████████████████
  autoenc      13/86  (15.1%)  ███████
  dictlearn     5/86  ( 5.8%)  ██

Saved → c:\MLResearch\outputs\figures\method_wins.png


## Analysis 3: Marginal Contribution to Oracle

For each method m, compute:
`marginal(m) = mean(oracle_LSE_with_all_6) - mean(oracle_LSE_without_m)`

A method with marginal contribution < 0.01 mean LSE can be removed with negligible impact.

In [4]:
oracle_all = df[LSE_COLS].max(axis=1).mean()
print(f'Oracle (all 6 methods): {oracle_all:.4f}')

marginal = {}
for m, col in zip(METHOD_NAMES, LSE_COLS):
    remaining_cols = [c for c in LSE_COLS if c != col]
    oracle_without = df[remaining_cols].max(axis=1).mean()
    marginal[m] = round(oracle_all - oracle_without, 4)

print('\n=== Marginal contribution to oracle ===')
print(f'  (oracle with all 6 = {oracle_all:.4f})')
for m, contrib in sorted(marginal.items(), key=lambda x: x[1], reverse=True):
    tag = ' ← candidate for removal' if contrib < 0.01 else ''
    print(f'  {m:10s}  +{contrib:.4f}{tag}')

fig, ax = plt.subplots(figsize=(8, 4))
names  = list(marginal.keys())
values = list(marginal.values())
colors = ['#4C72B0' if v >= 0.01 else '#DD8452' for v in values]
ax.bar(names, values, color=colors)
ax.axhline(0.01, ls='--', color='red', lw=1, label='0.01 threshold')
ax.set_ylabel('Marginal contribution to oracle (mean LSE)')
ax.set_title('Method Marginal Contribution\n(orange = < 0.01 — candidate for removal)')
ax.legend()
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'method_marginal.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'\nSaved → {path}')

Oracle (all 6 methods): 0.7980

=== Marginal contribution to oracle ===
  (oracle with all 6 = 0.7980)
  gmm         +0.0270
  autoenc     +0.0084 ← candidate for removal
  kmeans      +0.0069 ← candidate for removal
  dictlearn   +0.0036 ← candidate for removal
  dbscan      +0.0035 ← candidate for removal
  agg         +0.0034 ← candidate for removal

Saved → c:\MLResearch\outputs\figures\method_marginal.png


## Summary & Decision

Review the three analyses above before proceeding. Guiding rules:
- **Keep** if the method wins on ≥ 5 datasets OR contributes ≥ 0.01 to oracle.
- **Remove candidate** if it is both rarely winning AND has low marginal contribution.

The final decision must be made manually based on the printed results.

In [5]:
summary = pd.DataFrame({
    'wins'       : wins,
    'wins_pct'   : wins_pct,
    'marginal'   : pd.Series(marginal),
}).reindex(METHOD_NAMES)

summary['keep'] = (summary['wins'] >= 5) | (summary['marginal'] >= 0.01)

print('=== Method Retention Summary ===')
print(summary.to_string())
print()
print('Methods recommended for removal (fails BOTH thresholds):')
candidates = summary[~summary['keep']].index.tolist()
print(' ', ', '.join(candidates) if candidates else 'None')
print()
print('Note: proceed to 04_metafeatures.ipynb with the chosen method set.')
summary.to_csv(os.path.join(META_DIR, 'method_redundancy_summary.csv'))
print(f'Summary saved → {os.path.join(META_DIR, "method_redundancy_summary.csv")}')

=== Method Retention Summary ===
           wins  wins_pct  marginal  keep
kmeans       26      30.2    0.0069  True
dbscan        6       7.0    0.0035  True
agg           7       8.1    0.0034  True
gmm          29      33.7    0.0270  True
autoenc      13      15.1    0.0084  True
dictlearn     5       5.8    0.0036  True

Methods recommended for removal (fails BOTH thresholds):
  None

Note: proceed to 04_metafeatures.ipynb with the chosen method set.
Summary saved → c:\MLResearch\data\meta_table\method_redundancy_summary.csv
